# 04 — Bayesian fitting with DREAM (Student workbook)

**Time:** 60–75 minutes (advanced extension)
**Question:** what can a posterior distribution tell us that a single best fit
cannot?

Earlier notebooks returned one optimum and a local uncertainty estimate. Here
you will sample a distribution of plausible parameter combinations using the
DREAM Markov chain Monte Carlo algorithm.

> **Lecture connection**
>
> - Slide 34, *Deeper into uncertainty: priors and bounds*: every fit has
>   priors (the bounds, the model, the q range you kept); a bound is a uniform
>   prior; a posterior pressed against a bound means the bound, not the data,
>   chose the answer; a restricted q range destroys information. This notebook
>   lets you test each of these on the sphere from notebook 01.
> - Slide 35, *Wrap-up for part 2*: "several of these parameters are
>   degenerate ($\phi \leftrightarrow \Delta\rho \leftrightarrow$ scale)". Here
>   you will *see* a degeneracy as a tilted cloud in a pair plot.
> - Slide 20's discussion: concentration and contrast move the curve up and
>   down, while size moves it left and right. Keep that picture in mind when
>   you predict the correlations below.
> - Notebook 02's non-identifiability is the extreme case of what you measure
>   here: a posterior that is long and flat in some direction.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
DATA = ROOT / 'data'

# This workshop uses sans-fitter 0.4 or newer (pip install "sans-fitter>=0.4").
print(f'Using sans-fitter {sans_fitter.__version__}')
major, minor = (int(part) for part in sans_fitter.__version__.split('.')[:2])
assert (major, minor) >= (0, 4), 'Please install sans-fitter 0.4 or newer'
%matplotlib inline


In [ ]:
filename = DATA / 'sphere_reference.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## 1 — From bounds to priors

Bayes' theorem combines a likelihood with prior information:

$$
p(\theta \mid D) \propto p(D \mid \theta)\,p(\theta).
$$

For this workshop, `set_param(..., min=a, max=b, vary=True)` defines a
**uniform prior** between `a` and `b`: every value inside is initially equally
plausible and values outside are forbidden. This is not merely an optimizer
safety rail. Bounds must therefore be scientifically defensible.

Configure the same sphere model as in notebook 01. Keep the SLDs fixed and give
radius, scale and background bounded uniform priors.

**Where the likelihood comes from.** If each measured point is Gaussian with
standard deviation `dI`, then
$p(D \\mid \\theta) \\propto \\exp(-\\chi^2(\\theta)/2)$, with the same $\\chi^2$ you
minimized in notebook 01. The best fit is the peak of this function. Sampling
maps out its whole shape. This is why correct error bars from data reduction
(slides 44–45) matter even more here than for a point estimate.

**Fixing the SLDs is itself a prior.** It is a delta-function prior that
breaks the $\\phi \\leftrightarrow \\Delta\\rho$ degeneracy of slide 35 by fiat.
Ask yourself how confident you really are in those numbers (slide 12's
caveats about density and exchangeable hydrogens).

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(filename))
fitter.set_resolution('none')

# YOUR CODE HERE:
#   1. fitter.set_model('sphere') and fitter.get_params()
#   2. Fix both SLDs (4.2 and 1.0) with vary=False.
#   3. Give radius, scale and background a value, min, max and vary=True.
#      The min-max range is now your uniform prior. Keep it defensible but not
#      so broad that the short classroom run cannot explore it.
# Delete the `raise` line below when your lines are in place.
raise NotImplementedError('Configure the sphere model and its priors, then delete this line')


## 2 — Sampling with DREAM

`fit_bayesian` uses several controls:

- `samples`: approximate posterior sampling budget;
- `burn`: early generations discarded while chains approach high probability;
- `thin`: retain every nth draw;
- `pop`: population size multiplier for DREAM's parallel chains.

A larger budget is not automatically trustworthy. Chains must mix, explore the
same region, and be insensitive to starting values. The short run below is
chosen for class time and is **exploratory, not publication quality**.

DREAM is a population method, like the differential-evolution optimizer from
the notebook 01 extension. The difference is that it sometimes *accepts worse*
parameter sets, in proportion to their probability, so the chains end up
visiting regions in proportion to their posterior probability instead of
converging to a single point ([BUMPS optimizer guide: DREAM](https://bumps.readthedocs.io/en/latest/guide/optimizer.html#dream)).

In [ ]:
# YOUR CODE HERE: two lines.
#   1. fitter.fit_bayesian(samples=1500, burn=100, thin=1, pop=10)
#   2. fitter.plot_results(show_residuals=True, log_scale=True)
# The seed below makes everyone's chains identical; leave it in place.
np.random.seed(2026)
# Delete the `raise` line below when your lines are in place.
raise NotImplementedError('Run DREAM and plot the result, then delete this line')


## 3 — Posterior data and convergence

`get_posterior()` returns:

- `labels` and a two-dimensional `samples` array;
- 68% and 95% credible intervals;
- separate `chains` where available;
- R-hat diagnostics comparing within-chain and between-chain variation.

R-hat near 1 is desirable. Values above roughly 1.05 are a warning to run
longer, revisit priors, or investigate a multimodal posterior. Trace plots
should look stationary and well mixed rather than drifting or separating.

In [ ]:
posterior = fitter.get_posterior()
print('Parameters:', posterior.labels)
print('Combined sample shape (draws, parameters):', posterior.samples.shape)

# For each parameter: the 95 % credible interval and the R-hat convergence check
# (close to 1 is good; above about 1.05 means the chains disagree).
for name in posterior.labels:
    low, high = posterior.ci_95[name]
    r_hat = posterior.diagnostics.get(name, {}).get('r_hat')
    r_hat_text = f'{r_hat:.3f}' if r_hat is not None else 'n/a'
    print(f'{name:>12}: 95% CI [{low:.6g}, {high:.6g}], R-hat={r_hat_text}')

# Trace plots: each chain's value against step number. Look for flat, well-mixed
# "fuzzy caterpillars" rather than drifts or chains that stay apart.
fitter.plot_trace()


## 4 — Marginal distributions and correlations

A marginal distribution shows uncertainty in one parameter after averaging
over the others. A pair plot and correlation matrix reveal combinations that
the data constrain together. For example, particle contrast, concentration,
and scale can compensate for one another.

**Predict before plotting.** With both SLDs fixed, the plateau height is
$I(0) \propto \mathrm{scale}\cdot V \propto \mathrm{scale}\cdot r^3$ (slide 20).
If the data pin $I(0)$ tightly, which sign of correlation do you expect between
`scale` and `radius`? Is it strong, given that the *positions* of the minima
pin the radius independently (slide 23)? Which parameter should be most
correlated with `background`, and in which q region do they compete?

In [ ]:
# YOUR CODE HERE: one line, the marginal distribution of the radius:
#   fitter.plot_param_distribution('radius')
# Delete the `raise` line below when your line is in place.
raise NotImplementedError('Plot the radius distribution, then delete this line')


In [ ]:
# YOUR CODE HERE: one line, the pair plot of all sampled parameters:
#   fitter.plot_posterior_pairs()
# Delete the `raise` line below when your line is in place.
raise NotImplementedError('Plot the posterior pairs, then delete this line')


In [ ]:
# YOUR CODE HERE: one line, the correlation matrix (-1 to 1):
#   fitter.plot_param_correlations()
# Delete the `raise` line below when your line is in place.
raise NotImplementedError('Plot the parameter correlations, then delete this line')


## 5 — Posterior predictive checking

Parameter intervals are not the final scientific target. A posterior
predictive check propagates sampled parameters through the scattering model.
The resulting band asks whether plausible parameter sets reproduce the
observations across q, and where predictions remain uncertain.

Look especially at the form-factor minima and at the highest q, where the
signal approaches the background. Relate what you see to the lecture's point
that resolution and polydispersity fill in minima (slide 23). If those effects
were present but missing from the model, would the posterior band be wider,
narrower, or simply in the wrong place?

In [ ]:
# YOUR CODE HERE: one line.
#   fitter.plot_posterior_predictive(style='band+draws', n_draws=40)
# It draws parameter sets from the posterior, computes the model for each, and
# shows the 95 % band they span plus 40 individual curves.
# Delete the `raise` line below when your lines are in place.
raise NotImplementedError('Plot the posterior predictive check, then delete this line')


## 6 — Interpretation

Write a short report addressing:

1. Which parameter is best constrained, and what feature of the curve does it control?
2. Which parameters are correlated, and why?
3. Do the traces and R-hat values support convergence?
4. Where is the predictive band widest?
5. What would you change before publishing these intervals?

**Your answer** *(double-click this cell to edit it, then press Shift + Enter)*

Summarize what DREAM adds beyond the best fit, answering questions 1–5 above. Refer to the credible intervals, the trace plots, the correlations, and the predictive band.

## Optional export

The raw chain can be analyzed with other tools:

```python
posterior.save_posterior_csv('sphere_posterior.csv')
fitter.save_results('sphere_bayesian_fit.csv')
```

Try changing one prior range or restricting q. Rerun the sampler and explain
whether the posterior changed because of the data or because of the prior.

Two experiments from slide 34, each a one-line change before rerunning the
sampler cell:

1. *A bound that chooses the answer.* Cap the radius below its true value with
   `fitter.set_param('radius', max=55)`. Does χ² warn you? Where does the
   radius posterior pile up?
2. *Restricted q destroys information.* Keep only `0.004–0.025 Å⁻¹` with
   `fitter.set_q_range(qmin=0.004, qmax=0.025)`. Which posterior widens most,
   and does the pair plot now show a "banana" between size and scale?

## Further reading

- [BUMPS optimizer guide](https://bumps.readthedocs.io/en/latest/guide/optimizer.html): DREAM, differential
  evolution, and how BUMPS reports uncertainties.
- J. A. Vrugt, *Markov chain Monte Carlo simulation using the DREAM software
  package*, Environ. Model. Softw. 75, 273 (2016): the algorithm behind
  `fit_bayesian`.
- A. Vehtari et al., *Rank-normalization, folding, and localization: an improved
  R-hat for assessing convergence of MCMC*, Bayesian Anal. 16, 667 (2021): why
  R-hat thresholds have tightened from 1.1 to about 1.01.
- [Jacques & Trewhella, *Small-angle scattering for structural biology: expanding the frontier while avoiding the pitfalls*, Protein Sci. 19, 642 (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2867006/): on reporting uncertainty and model ambiguity in SAS results.